# FOV Finder Agent — standalone demo

Uses `FOVFinderAgent` on its own (no BO, no composition) to:

1. Load a plate calibration (`WellPlatePlan`) saved by the **MDA plate widget** of `pymmcore-widgets`.
2. For an ordered list of wells, generate random candidate positions inside each well — kept `border_um` away from the well edge.
3. Image each candidate via `mic.run_mda` with the user's imaging channels.
4. Segment each frame with the user's `Segmentator` and count cells.
5. Filter out positions with `< min_cells` cells, then pick `fovs_per_well` per well via **farthest-point sampling** so they don't overlap.

This notebook is the simplest possible usage. See `experiments/31_bo_optimisation/bo_erk_oscillation.ipynb` for the FOV finder coupled to a BO inter-phase agent via `ComposedAgent`.

## 1. Microscope and channel setup

In [ ]:
from faro.microscope.pertzlab.jungfrau import Jungfrau
from faro.core.data_structures import PowerChannel

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

# Channels used to scan candidate positions. Stim/ref channels are intentionally
# *not* used here — the FOV finder only acquires imaging channels.
imaging_channels = (
    PowerChannel(config="miRFP", exposure=125, group="TTL_ERK", power=95),
    PowerChannel(config="mScarlet3", exposure=125, group="TTL_ERK", power=95),
)

## 2. Plate calibration

Calibrate the plate once using the `pymmcore-widgets` MDA plate widget (it stores the affine transformation as `a1_center_xy` + `rotation` in a `useq.WellPlatePlan` JSON file). Point this notebook at that file.

In [ ]:
PLATE_CALIBRATION_PATH = r"E:\Alex\plate_calibration_96well.json"  # <-- UPDATE

# Quick sanity check
from useq import WellPlatePlan

plan = WellPlatePlan.from_file(PLATE_CALIBRATION_PATH)
print(f"plate: {plan.plate.name}  rows={plan.plate.rows}  cols={plan.plate.columns}")
print(f"well_size mm: {plan.plate.well_size}  circular: {plan.plate.circular_wells}")
print(f"a1_center_xy: {plan.a1_center_xy}  rotation: {plan.rotation}")

## 3. Segmentator

Use whichever segmentator gives a sensible cell count from the channel you want to score (here we segment the first imaging channel — the nuclear marker).

In [ ]:
from faro.segmentation.cellpose_v4 import CellposeV4

segmentator = CellposeV4()

## 4. Configure the FOV finder

Key knobs:

- `wells` — ordered list of wells (e.g. `["B2", "B3", "B4", ...]`).
- `wells_per_phase` — how many wells one `run()` call consumes.
- `fovs_per_well` — how many positions per well are returned.
- `n_candidates_per_well` — how many random candidates are scanned per well; should be larger than `fovs_per_well` so the farthest-point sampler has room to spread the picks.
- `border_um` — minimum distance to keep candidates away from the well edge.
- `min_cells` — minimum cell count for a candidate to be considered valid.
- `seg_channel_index` — which imaging channel to segment (default `0`).
- `feature_extractor` — optional; if supplied, its `extract_positions({"labels": label_image})` is called per FOV and the results are attached to `all_candidates` (purely informational, does not influence selection).

In [ ]:
from faro.agents import FOVFinderAgent

# Example: 6 wells in row B, scan 8 candidates per well, return 3 per well
wells = ["B2", "B3", "B4", "B5", "B6", "B7"]

finder = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=wells,
    wells_per_phase=6,
    fovs_per_well=3,
    n_candidates_per_well=8,
    border_um=400.0,  # keep at least 400 µm from the well edge
    min_cells=20,  # need >=20 cells in the seg channel
    imaging_channels=imaging_channels,
    segmentator=segmentator,
    seg_channel_index=0,  # segment on miRFP (nuclear marker)
    random_seed=42,
)

print(f"Wells queued: {len(finder.remaining_wells)}")
print(f"Phases this queue can drive: {finder.n_remaining_phases}")

## 5. Run one phase

`run()` consumes the next `wells_per_phase` wells, scans, segments, filters and selects FOVs.

In [ ]:
result = finder.run()

selected = result["positions"]
wells_used = result["wells_used"]
df_candidates = result["all_candidates"]

print(f"Phase {result['phase']} — wells: {wells_used}")
print(f"Selected {len(selected)} FOVs")
print(f"Scanned {len(df_candidates)} candidate positions")
print(f"\nValid candidates per well:")
print(df_candidates.groupby("well")["valid"].sum())

for fp in selected:
    print(f"  {fp.name}: x={fp.x:.1f}  y={fp.y:.1f}  z={fp.z}")

## 6. Visualise candidates per well

In [ ]:
import matplotlib.pyplot as plt

# Map (well -> selected positions in that well) using the parallel
# wells_for_positions list returned by the FOV finder.
wells_for_positions = result["wells_for_positions"]
selected_by_well = {}
for fp, w in zip(selected, wells_for_positions):
    selected_by_well.setdefault(w, []).append(fp)

fig, axes = plt.subplots(
    1, len(wells_used), figsize=(4 * len(wells_used), 4), squeeze=False
)
for ax, well in zip(axes[0], wells_used):
    df_w = df_candidates[df_candidates["well"] == well]
    ax.scatter(
        df_w["x"],
        df_w["y"],
        c=df_w["n_cells"],
        cmap="viridis",
        s=80,
        edgecolors="k",
        linewidths=0.5,
    )
    # Highlight selected
    sel = selected_by_well.get(well, [])
    if sel:
        ax.scatter(
            [fp.x for fp in sel],
            [fp.y for fp in sel],
            marker="x",
            c="red",
            s=200,
            linewidths=3,
            label="selected",
        )
    ax.set_title(f"well {well}")
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    ax.set_aspect("equal")
    ax.legend(loc="best")
plt.tight_layout()
plt.show()

## 7. (Optional) drive multiple phases

Each call to `run()` consumes the next `wells_per_phase` wells from the queue. You can call `run()` repeatedly to step through batches of wells.

In [ ]:
# Example: another finder with smaller batches so we can run multiple phases
finder2 = FOVFinderAgent(
    microscope=mic,
    well_plate_plan=PLATE_CALIBRATION_PATH,
    wells=["C2", "C3", "C4", "C5"],
    wells_per_phase=2,
    fovs_per_well=2,
    n_candidates_per_well=6,
    border_um=400.0,
    min_cells=20,
    imaging_channels=imaging_channels,
    segmentator=segmentator,
    random_seed=7,
)

all_phases = []
while finder2.remaining_wells:
    res = finder2.run()
    all_phases.append(res)
    print(
        f"phase {res['phase']}: {len(res['positions'])} FOVs from wells {res['wells_used']}"
    )

print(f"\nTotal phases run: {len(all_phases)}")